In [10]:
from pathlib import Path
import pandas as pd
import altair as alt

# Load master dataset compiled from regression notebook
DATASET_PATH = Path("../data/italy_master_file_generated_from_regression_ipynb.json")

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at: {DATASET_PATH.resolve()}")

italy_df = pd.read_json(DATASET_PATH)
italy_df.head()

,TIME_PERIOD,Total_Accidents,Territory,Population,Accidents_Per_100k,Parking difficulty (very),Parking difficulty (very & quite),Poor public transport (very),Poor public transport (very & quite),Poor road conditions (very),...,Household income average (excl. imputed rent),GDP per capita (EUR),Alcohol/tobacco spend (% of total),Alcohol/tobacco spend per capita (EUR),Gov. exp. total per capita (EUR),Gov. exp. public order per capita (EUR),Gov. exp. economic affairs per capita (EUR),Gov. exp. health per capita (EUR),Gov. exp. education per capita (EUR),Gov. exp. social protection per capita (EUR)
0,2010,420754,All Regions,59816673,703.405888,18.7,39.6,10.3,29.5,23.0,...,30220.0,30441.853561,4.283121,785.842770,6085.075979,563.683975,453.223134,2034.452836,1160.776695,313.382525
1,2011,405294,All Regions,59816673,677.560251,17.9,38.0,9.8,28.6,21.7,...,30236.0,30653.565437,4.296078,790.565533,5948.107478,569.939756,463.621573,2000.631162,1100.524264,292.268010
2,2012,369928,All Regions,59816673,618.436268,14.8,35.8,9.9,28.8,17.9,...,29579.0,29695.568324,4.460336,792.635191,5809.373584,538.588965,441.139546,1953.595781,1082.507548,287.085509
3,2013,356982,All Regions,59816673,596.793473,15.8,37.2,10.5,31.3,22.6,...,29473.0,29155.695436,4.427740,769.773672,5746.730514,540.832487,433.957602,1930.104003,1078.117467,280.744467
4,2014,348058,All Regions,59816673,581.874555,14.9,35.2,10.2,30.7,21.2,...,29472.0,29155.290867,4.232236,737.699002,5712.243474,535.992699,420.658635,1956.616678,1076.922148,273.002479


## Handling region aggregates
To keep region and time fixed effects clean, we should drop the aggregate `All Regions` rows (they pool all regions and would otherwise double-count observations). We'll filter them out and keep the disaggregated territories only.

In [11]:
# Drop the national aggregate to avoid double-counting in region fixed effects
italy_df = italy_df[italy_df["Territory"] != "All Regions"].copy()

# Make sure panel identifiers are typed correctly
italy_df["TIME_PERIOD"] = italy_df["TIME_PERIOD"].astype(int)
italy_df["Territory"] = italy_df["Territory"].astype("category")

print("Post-filter shape:", italy_df.shape)
print("Remaining territories:", italy_df["Territory"].cat.categories.tolist())

Post-filter shape: (300, 50)
Remaining territories: ['Abruzzo', 'Basilicata', 'Calabria', 'Campania', 'Emilia-Romagna', 'Friuli-Venezia Giulia', 'Lazio', 'Liguria', 'Lombardia', 'Marche', 'Molise', 'Piemonte', 'Puglia', 'Sardegna', 'Sicilia', 'Toscana', 'Trentino Alto Adige / Südtirol', 'Umbria', "Valle d'Aosta / Vallée d'Aoste", 'Veneto']


In [12]:
# Inspect panel time coverage
print("TIME_PERIOD min/max:", italy_df["TIME_PERIOD"].min(), italy_df["TIME_PERIOD"].max())
unique_periods = sorted(italy_df["TIME_PERIOD"].unique())
print("Unique TIME_PERIOD values (sorted):", unique_periods)
print("Count of periods:", len(unique_periods))

TIME_PERIOD min/max: 2010 2024
Unique TIME_PERIOD values (sorted): [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Count of periods: 15


In [13]:
# Prepare numeric panel data (coerce objects to numeric, drop rows with missing)
target = "Accidents_Per_100k"
cols = italy_df.columns.tolist()
start_idx = cols.index(target)

# Baseline controls (all to the right of target) before any pruning
base_control_cols = cols[start_idx + 1 :]

# Remove high-correlation/problematic variables from controls
removed_controls = [
    'Household income median (incl. imputed rent)',
    'Household income average (incl. imputed rent)',
    'Household income median (excl. imputed rent)',
    'Household income average (excl. imputed rent)',
    'Commute using any transport (share)',
    'Alcohol/tobacco spend (% of total)',
    'Gov. exp. total per capita (EUR)',
    'Total vehicles per capita (fleet)'
]
control_cols = [
    c for c in base_control_cols
    if not c.endswith('(very & quite)')
    and c not in removed_controls
]

# Coerce numerics for full and pruned sets separately
numeric_cols_full = [target] + base_control_cols
italy_num = italy_df.copy()
for col in numeric_cols_full:
    italy_num[col] = pd.to_numeric(italy_num[col], errors="coerce")

# Drop rows with missing target/controls plus FE identifiers to avoid mixed group dtypes later
required_full = [target] + base_control_cols + ["TIME_PERIOD", "Territory"]
required_pruned = [target] + control_cols + ["TIME_PERIOD", "Territory"]
italy_clean_full = italy_num.dropna(subset=required_full).copy()
italy_clean = italy_num.dropna(subset=required_pruned).copy()

print("Original rows:", len(italy_df))
print("Rows after dropna on full set (target+controls+FE ids):", len(italy_clean_full))
print("Rows after dropna on pruned set (target+controls+FE ids):", len(italy_clean))
print("Any object dtypes remaining in pruned?:", italy_clean.dtypes.eq("object").any())
print("Controls used ({}):".format(len(control_cols)))
print(control_cols)

Original rows: 300
Rows after dropna on full set (target+controls+FE ids): 280
Rows after dropna on pruned set (target+controls+FE ids): 280
Any object dtypes remaining in pruned?: False
Controls used (32):
['Parking difficulty (very)', 'Poor public transport (very)', 'Poor road conditions (very)', 'Poor street lighting (very)', 'Traffic problems (very)', 'Commute under 15 min (share)', 'Commute 31+ min (share)', 'Commute by bicycle (share)', 'Commute by tram or city bus (share)', 'Commute by bus company/charter (share)', 'Commute by coach (share)', 'Commute on foot (share)', 'Commute by metro (share)', 'Commute by motorcycle or moped (share)', 'Commute as car driver (share)', 'Commute as car passenger (share)', 'Commute by train (share)', 'Buses or trolley-buses per capita (vehicles)', 'Motor cars per capita (vehicles)', 'Motor units per capita (vehicles)', 'Motorcycles per capita (vehicles)', 'Other vehicles per capita (vehicles)', 'Three-wheelers or motor-vans per capita (vehicles)'

In [14]:
# Set reference categories for fixed effects: Lazio (region) and 2019 (year)
years = sorted(italy_clean_full["TIME_PERIOD"].dropna().unique())
years = [2019] + [y for y in years if y != 2019]
for df in (italy_clean, italy_clean_full):
    df["TIME_PERIOD"] = pd.Categorical(df["TIME_PERIOD"], categories=years, ordered=True)

regions = sorted(italy_clean_full["Territory"].astype(str).unique())
regions = ["Lazio"] + [r for r in regions if r != "Lazio"]
for df in (italy_clean, italy_clean_full):
    df["Territory"] = pd.Categorical(df["Territory"], categories=regions, ordered=True)

print("FE reference categories set for both pruned and full datasets: TIME_PERIOD base=2019, Territory base=Lazio")

FE reference categories set for both pruned and full datasets: TIME_PERIOD base=2019, Territory base=Lazio


## Correlation and VIF diagnostics for controls
Run this before the FE OLS to spot redundancy. Order to run: Cells 1→5 (prep), then this section (Cells 7), then baseline FE OLS (Cells 9), then LASSO/ENet and post-selection (Cells 11–12).

In [6]:
import itertools
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Correlation matrix for controls
df_ctrl = italy_clean[control_cols]
corr = df_ctrl.corr().abs()
# Extract top absolute correlations (upper triangle)
pairs = []
for i, j in itertools.combinations(range(corr.shape[0]), 2):
    pairs.append((corr.iloc[i, j], corr.index[i], corr.columns[j]))
top_corr = sorted(pairs, key=lambda x: x[0], reverse=True)[:25]

print("Top 25 absolute correlations among controls:")
for val, c1, c2 in top_corr:
    print(f"{val:.3f} : {c1} vs {c2}")

# VIF for controls (no intercept added here; add_constant inside VIF loop)
X_vif = sm.add_constant(df_ctrl)
vif = []
for i, col in enumerate(X_vif.columns):
    if col == "const":
        continue
    vif_val = variance_inflation_factor(X_vif.values, i)
    vif.append((col, vif_val))

vif_sorted = sorted(vif, key=lambda x: x[1], reverse=True)
print("\nTop 20 VIFs (controls only):")
for col, v in vif_sorted[:20]:
    print(f"{col}: {v:.2f}")

Top 25 absolute correlations among controls:
0.972 : Motor cars per capita (vehicles) vs Trucks per capita (vehicles)
0.897 : Motor units per capita (vehicles) vs Trailers per capita (vehicles)
0.875 : Commute by motorcycle or moped (share) vs Motorcycles per capita (vehicles)
0.853 : Commute 31+ min (share) vs Commute by metro (share)
0.830 : Three-wheelers or motor-vans per capita (vehicles) vs Gov. exp. economic affairs per capita (EUR)
0.811 : Trucks per capita (vehicles) vs Gov. exp. economic affairs per capita (EUR)
0.810 : Commute by tram or city bus (share) vs Commute by train (share)
0.800 : Gov. exp. public order per capita (EUR) vs Gov. exp. social protection per capita (EUR)
0.791 : Motor cars per capita (vehicles) vs Gov. exp. economic affairs per capita (EUR)
0.787 : Poor road conditions (very) vs Poor street lighting (very)
0.773 : Three-wheelers or motor-vans per capita (vehicles) vs Trucks per capita (vehicles)
0.761 : Gov. exp. economic affairs per capita (EUR) vs Gov

## Correlation heatmap (controls only, Altair)
Interactive heatmap of pairwise correlations among controls; hover for values. Self-correlations are omitted.

In [7]:
# Correlation heatmaps with and without "(very & quite)" variables
controls_initial = base_control_cols
controls_pruned = control_cols

# Use full dataset for initial (full controls) and pruned dataset for pruned view
corr_before = italy_clean_full[controls_initial].corr()
corr_after = italy_clean[controls_pruned].corr()

def corr_to_long(corr_df, label):
    return (
        corr_df.reset_index()
        .melt(id_vars='index', var_name='variable', value_name='correlation')
        .query('index != variable')
        .assign(view=label)
    )

corr_before_long = corr_to_long(corr_before, 'Initial')
corr_after_long = corr_to_long(corr_after, 'After Variable Pruning')

corr_compare = pd.concat([corr_before_long, corr_after_long], ignore_index=True)

view_param = alt.param(
    name='view_select',
    value='Initial',
    bind=alt.binding_select(options=['Initial', 'After Variable Pruning'], name='Heatmap view: ')
)

heatmap_compare = (
    alt.Chart(corr_compare)
    .add_params(view_param)
    .transform_filter(alt.datum.view == view_param)
    .mark_rect()
    .encode(
        x=alt.X('variable:O', sort=list(corr_before.columns), title=''),
        y=alt.Y('index:O', sort=list(corr_before.index), title=''),
        color=alt.Color('correlation:Q', scale=alt.Scale(scheme='blueorange', domain=[-1, 1])),
        tooltip=['index', 'variable', alt.Tooltip('correlation:Q', format='.3f'), 'view']
    )
    .properties(width=500, height=500)
)

heatmap_compare

alt.Chart(...)

## Baseline two-way fixed effects OLS
We residualize via dummies for time and territory (dropping one category each) and cluster standard errors by territory. Controls are all columns to the right of `Accidents_Per_100k`.

In [15]:
import statsmodels.api as sm

# Use cleaned data and controls from prior step (pruned set)
X_base = italy_clean[control_cols].copy()

# Add time and territory fixed effects via dummies (drop_first avoids multicollinearity)
dummies_time = pd.get_dummies(italy_clean["TIME_PERIOD"], prefix="T", drop_first=True)
dummies_region = pd.get_dummies(italy_clean["Territory"], prefix="R", drop_first=True)

X = pd.concat([X_base, dummies_time, dummies_region], axis=1)

# Detect and fix any object dtypes that slipped through
obj_cols = X.select_dtypes(include=["object"]).columns.tolist()
if obj_cols:
    print("Object columns found, coercing to numeric:", obj_cols)
    X[obj_cols] = X[obj_cols].apply(pd.to_numeric, errors="coerce")

X = X.astype(float)
X = sm.add_constant(X)

y = italy_clean[target].astype(float)

# Cluster groups: ensure pure string to avoid mixed-type compare errors
cluster_groups = italy_clean["Territory"].astype(str).to_numpy()

model_fe = sm.OLS(y, X).fit(cov_type="cluster", cov_kwds={"groups": cluster_groups})

print("Design matrix shape:", X.shape)
print(model_fe.summary().tables[1])
print("\nR-squared:", model_fe.rsquared)

Design matrix shape: (280, 65)
                                                         coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------------------------
const                                               1771.5341    415.628      4.262      0.000     956.918    2586.150
Parking difficulty (very)                              0.7509      1.603      0.468      0.639      -2.391       3.893
Poor public transport (very)                          -0.7599      1.373     -0.554      0.580      -3.450       1.930
Poor road conditions (very)                           -2.3563      1.022     -2.306      0.021      -4.359      -0.354
Poor street lighting (very)                            2.3119      1.999      1.157      0.247      -1.606       6.230
Traffic problems (very)                                2.5116      1.739      1.444      0.149      -0.896       5.920
Commute under 15 

c:\Users\juanx\anaconda3\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 64, but rank is 19
  warnings.warn('covariance of constraints does not have full '


In [9]:
# Predicted vs actual and residuals vs predicted (toggle view)
df_pred = pd.DataFrame({
    'y_pred': model_fe.fittedvalues,
    'y_actual': italy_clean[target].astype(float),
    'residual': model_fe.resid,
})

min_xy = df_pred[['y_pred', 'y_actual']].min().min()
max_xy = df_pred[['y_pred', 'y_actual']].max().max()
line_df_45 = pd.DataFrame({
    'x': [min_xy, max_xy],
    'y': [min_xy, max_xy],
    'view': ['Actual vs Predicted', 'Actual vs Predicted']
})

x_min = df_pred['y_pred'].min()
x_max = df_pred['y_pred'].max()
line_df_zero = pd.DataFrame({
    'x': [x_min, x_max],
    'y': [0.0, 0.0],
    'view': ['Residuals vs Predicted', 'Residuals vs Predicted']
})

df_long = pd.concat([
    df_pred.assign(view='Actual vs Predicted', y=df_pred['y_actual']),
    df_pred.assign(view='Residuals vs Predicted', y=df_pred['residual']),
], ignore_index=True)

view_select = alt.param(
    name='view_select',
    value='Actual vs Predicted',
    bind=alt.binding_select(options=['Actual vs Predicted', 'Residuals vs Predicted'], name='View: ')
)

points = (
    alt.Chart(df_long)
    .add_params(view_select)
    .transform_filter(alt.datum.view == view_select)
    .mark_circle(color='royalblue', opacity=0.6)
    .encode(
        x=alt.X('y_pred:Q', title='Predicted y'),
        y=alt.Y('y:Q', title='Actual y (or Residual)'),
        tooltip=['y_pred:Q', 'y_actual:Q', 'residual:Q']
    )
    .properties(width=450, height=400)
    .interactive()
)

line_45 = (
    alt.Chart(line_df_45)
    .add_params(view_select)
    .transform_filter(alt.datum.view == view_select)
    .mark_line(color='red', strokeDash=[4,4])
    .encode(x='x:Q', y='y:Q')
)

line_zero = (
    alt.Chart(line_df_zero)
    .add_params(view_select)
    .transform_filter(alt.datum.view == view_select)
    .mark_line(color='red', strokeDash=[4,4])
    .encode(x='x:Q', y='y:Q')
)

chart_pred_resid = points + line_45 + line_zero
chart_pred_resid

alt.LayerChart(...)

In [16]:
# Extract region fixed effects; keep significant (p<0.05) else set to 0; include baseline as 0
alpha = 0.05
region_prefix = "R_"
region_params = model_fe.params[model_fe.params.index.str.startswith(region_prefix)]
region_pvals = model_fe.pvalues.loc[region_params.index]
region_effects = region_params.where(region_pvals < alpha, other=0.0)

# Add baseline region (first in `regions`, e.g., Lazio) with zero by definition
baseline_region = regions[0] if regions else "baseline"
effects_df = pd.DataFrame({
    "Region": [baseline_region] + [name[len(region_prefix):] for name in region_effects.index],
    "Fixed_Effect": [0.0] + region_effects.values.tolist(),
})

# Save to data folder (JSON and Excel)
out_base = Path("../data/italy_accidents_significant_region_fixed_effects")
out_json = out_base.with_suffix(".json")
out_xlsx = out_base.with_suffix(".xlsx")
effects_df.to_json(out_json, orient="records", indent=2)
effects_df.to_excel(out_xlsx, index=False)
print("Saved:", out_json)
print("Saved:", out_xlsx)
effects_df.head()

Saved: ..\data\italy_accidents_significant_region_fixed_effects.json
Saved: ..\data\italy_accidents_significant_region_fixed_effects.xlsx


,Region,Fixed_Effect
0,Lazio,0.000000
1,Abruzzo,-240.297935
2,Basilicata,-367.188906
3,Calabria,0.000000
4,Campania,-359.648036
